Notebook 05: H1 Control Contribution and Sensitivity Analysis

H1 Control Contribution and Sensitivity Analysis

H1: Zero Trust implementation improves anomaly detection performance in RFID baggage tracking systems compared to traditional rule based methods.

RQ1: How can Zero Trust improve anomaly detection in RFID baggage tracking systems?

This notebook examines which Zero Trust checks contribute to H1 anomaly detection and how timing and duplicate scan settings affect performance. The prepared RFID data from Notebook 03 is reused without changes.

The tests use known simulated labels and additional controlled variations. They evaluate the implemented rules, not airport deployment or independent verification with new RFID data.

BLUF: The analysis examines control contribution, sensitivity to security settings, and possible false alerts when otherwise normal scans contain simulated timing variations.

Step 1: Connect and Review the H1 Research Data

The prepared RFID baggage tracking file is connected from the project folder. Record counts, labels, security fields, and the source file fingerprint are checked before testing.

In [1]:
# Step 1: Connect and review the H1 research data

# Import libraries used throughout the notebook.
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)

# Find the research project from Codespaces or its notebook folder.
main_folder = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "06_Prepared_Data").is_dir()),
    None
)
if main_folder is None:
    raise FileNotFoundError("Project folder with 06_Prepared_Data was not found.")

# Locate the original prepared data; do not edit or overwrite it.
prepared_file = main_folder / "06_Prepared_Data" / "02_Prepared_RFID_Data.csv"
if not prepared_file.is_file():
    raise FileNotFoundError(f"RFID dataset was not found: {prepared_file}")

# Load the prepared records for analysis.
h1_data = pd.read_csv(prepared_file)

# Confirm that the fields used in the original H1 security checks exist.
required_fields = [
    "bag_id", "tag_id", "reader_id", "reader_location",
    "custody_point", "timestamp", "flight_id", "route_id", "scan_id"
]
control_fields = [
    "expected_reader_id", "expected_timestamp", "location_match",
    "sequence_match", "delayed_indicator", "scan_gap_minutes",
    "final_label", "scan_status"
]
missing_fields = [
    name for name in required_fields + control_fields
    if name not in h1_data.columns
]
if missing_fields:
    raise ValueError(f"The prepared RFID file is missing: {missing_fields}")

# Confirm the original study size and two expected scan classes.
print("Prepared RFID file:", prepared_file)
print("Total RFID records:", len(h1_data))
print("Baggage journeys:", h1_data["bag_id"].nunique())
print("Total columns:", len(h1_data.columns))
print("\nScan labels:")
print(h1_data["final_label"].value_counts(dropna=False))

# Stop if this is not the dataset used for the original H1 comparison.
assert len(h1_data) == 31575, "Unexpected RFID record count."
assert h1_data["bag_id"].nunique() == 5000, "Unexpected journey count."
assert len(h1_data.columns) == 34, "Unexpected RFID column count."
assert h1_data["final_label"].value_counts().to_dict() == {
    "Normal": 30225, "Abnormal": 1350
}, "Unexpected original H1 scan labels."
assert h1_data["scan_id"].is_unique, "Scan IDs are not unique."

# Save a reproducible fingerprint of the input without changing the file.
source_sha256 = hashlib.sha256(prepared_file.read_bytes()).hexdigest()
print("\nSource SHA256:", source_sha256)
print("Step 1 complete. Original data unchanged.")

Prepared RFID file: /workspaces/Aviation-Baggage-Cybersecurity-Research/06_Prepared_Data/02_Prepared_RFID_Data.csv
Total RFID records: 31575
Baggage journeys: 5000
Total columns: 34

Scan labels:
final_label
Normal      30225
Abnormal     1350
Name: count, dtype: int64

Source SHA256: 33fa70a8cb17fd3109f9cd4618cd9f95a696dddd50c66fe4a8e6e9cbfd3a6848
Step 1 complete. Original data unchanged.


Step 2: Reproduce Traditional and Zero Trust Results

The original security checks are applied to the prepared RFID records. The analysis verifies the 825 traditional flags and 1,275 Zero Trust flags before changing any settings.

In [2]:
# Step 2: Reproduce traditional and Zero Trust results

# Preserve the original records and sort them by bag and scan time.
baseline_data = h1_data.copy()
baseline_data["timestamp"] = pd.to_datetime(
    baseline_data["timestamp"], errors="raise"
)
baseline_data["expected_timestamp"] = pd.to_datetime(
    baseline_data["expected_timestamp"], errors="raise"
)
baseline_data = baseline_data.sort_values(
    ["bag_id", "timestamp"], kind="mergesort"
).reset_index(drop=True)

# Define the recognized reader list used in Notebook 03.
recognized_readers = [
    "RDR001", "RDR002", "RDR003", "RDR004",
    "RDR005", "RDR006", "RDR007"
]

# Define a helper for security checks at chosen timing thresholds.
def get_control_checks(data, timing_limit=15.0, duplicate_limit=1.0):
    # Verify that the intended thresholds are not negative.
    if timing_limit < 0 or duplicate_limit < 0:
        raise ValueError("Control thresholds must be nonnegative.")

    # Get the preceding scan within each baggage journey.
    previous = data.groupby("bag_id", sort=False)[
        ["reader_id", "reader_location", "custody_point"]
    ].shift(1)

    # A duplicate uses the same reader, location, and custody point.
    # The original prepared scan gap is used to match Notebook 03.
    duplicate_found = (
        data["reader_id"].eq(previous["reader_id"])
        & data["reader_location"].eq(previous["reader_location"])
        & data["custody_point"].eq(previous["custody_point"])
        & pd.to_numeric(data["scan_gap_minutes"], errors="raise").le(duplicate_limit)
    )

    # Recalculate timing from the original actual and expected timestamps.
    # This lets Step 4 test alternative limits without editing source data.
    difference_minutes = (
        (data["timestamp"] - data["expected_timestamp"])
        .dt.total_seconds().abs() / 60
    )

    # True means that the scan passes the indicated security check.
    checks = pd.DataFrame({
        "Required information": data[required_fields].notna().all(axis=1),
        "Authorized reader": data["reader_id"].eq(data["expected_reader_id"]),
        "Location match": data["location_match"].eq(1),
        "Custody sequence": data["sequence_match"].eq(1),
        "Timing check": difference_minutes.le(timing_limit),
        "Duplicate check": ~duplicate_found,
    }, index=data.index)
    return checks.astype(bool)

# Set ground truth from the original simulated scan labels.
actual_abnormal = baseline_data["final_label"].eq("Abnormal")

# Apply the exact four conditions in Notebook 03's traditional decision.
traditional_prediction = ~(
    baseline_data[required_fields].notna().all(axis=1)
    & baseline_data["reader_id"].isin(recognized_readers)
    & baseline_data["location_match"].eq(1)
    & baseline_data["sequence_match"].eq(1)
)

# Apply the six checks from the original Zero Trust implementation.
control_checks = get_control_checks(baseline_data)
zta_prediction = ~control_checks.all(axis=1)

# Confirm calculated timing agrees with the saved Notebook 03 indicator.
assert control_checks["Timing check"].equals(
    baseline_data["delayed_indicator"].eq(0)
), "Timing calculation does not match the original prepared indicator."

# Define one consistent function for all reported detection measures.
def evaluate_predictions(name, actual, predicted):
    # Calculate the four confusion matrix categories.
    tn, fp, fn, tp = confusion_matrix(
        actual, predicted, labels=[False, True]
    ).ravel()

    # Return both counts and detection measures.
    return {
        "Scenario": name,
        "Records": int(len(actual)),
        "True Positives": int(tp),
        "True Negatives": int(tn),
        "False Positives": int(fp),
        "False Negatives": int(fn),
        "Accuracy": float(accuracy_score(actual, predicted)),
        "Precision": float(precision_score(actual, predicted, zero_division=0)),
        "Recall": float(recall_score(actual, predicted, zero_division=0)),
        "F1 Score": float(f1_score(actual, predicted, zero_division=0)),
    }

# Compare both original H1 rule sets.
baseline_comparison = pd.DataFrame([
    evaluate_predictions("Traditional rules", actual_abnormal, traditional_prediction),
    evaluate_predictions("Zero Trust", actual_abnormal, zta_prediction)
])
print("Original H1 performance:")
print(baseline_comparison.round(4).to_string(index=False))

# Require agreement with the verified Notebook 03 outputs.
assert int(traditional_prediction.sum()) == 825, "Traditional baseline differs from Notebook 03."
assert int(zta_prediction.sum()) == 1275, "Zero Trust baseline differs from Notebook 03."
assert int(baseline_comparison.loc[0, "True Positives"]) == 825
assert int(baseline_comparison.loc[1, "True Positives"]) == 1275
assert int(baseline_comparison.loc[1, "False Positives"]) == 0
print("\nStep 2 complete. Notebook 03 baseline reproduced.")

Original H1 performance:
         Scenario  Records  True Positives  True Negatives  False Positives  False Negatives  Accuracy  Precision  Recall  F1 Score
Traditional rules    31575             825           30225                0              525    0.9834        1.0  0.6111    0.7586
       Zero Trust    31575            1275           30225                0               75    0.9976        1.0  0.9444    0.9714

Step 2 complete. Notebook 03 baseline reproduced.


Step 3: Evaluate Individual Security Control Contributions

Each Zero Trust check is removed one at a time while the other checks remain active. Changes in detected abnormalities and false alerts are measured against the full Zero Trust decision. These results show the contribution of a control within the existing combined rule design; overlapping controls may identify the same scan.

In [3]:
# Step 3: Evaluate individual control contributions

# Evaluate the full Zero Trust set first.
full_result = evaluate_predictions("All six controls", actual_abnormal, zta_prediction)
control_rows = []

# Remove one security check at a time.
for removed_control in control_checks.columns:
    # Keep every check except the selected control.
    remaining = control_checks.drop(columns=[removed_control])

    # Flag scans that fail at least one remaining check.
    reduced_prediction = ~remaining.all(axis=1)

    # Evaluate the change in detection performance.
    row = evaluate_predictions(
        "Without " + removed_control,
        actual_abnormal,
        reduced_prediction
    )
    row["Removed Control"] = removed_control
    row["Fewer True Positives"] = (
        full_result["True Positives"] - row["True Positives"]
    )
    row["Change in False Positives"] = (
        row["False Positives"] - full_result["False Positives"]
    )
    control_rows.append(row)

# Present the control contribution results.
control_contribution = pd.DataFrame(control_rows)
print("H1 control contribution results:")
print(control_contribution[
    ["Removed Control", "True Positives", "False Negatives",
     "False Positives", "Fewer True Positives", "Recall"]
].round(4).to_string(index=False))
print("\nStep 3 complete.")

H1 control contribution results:
     Removed Control  True Positives  False Negatives  False Positives  Fewer True Positives  Recall
Required information            1275               75                0                     0  0.9444
   Authorized reader             900              450                0                   375  0.6667
      Location match            1050              300                0                   225  0.7778
    Custody sequence            1050              300                0                   225  0.7778
        Timing check            1050              300                0                   225  0.7778
     Duplicate check            1050              300                0                   225  0.7778

Step 3 complete.


Step 4: Test Security Control Sensitivity

The timing threshold and duplicate scan threshold are changed separately. The original 15-minute timing and one-minute duplicate thresholds remain the reference settings. The original labels are unchanged during these comparisons.

In [4]:
# Step 4: Test security control sensitivity

# Test timing windows around the original 15-minute threshold.
timing_thresholds = [5, 10, 15, 20, 25, 30]

# Test duplicate windows around the original one-minute threshold.
duplicate_thresholds = [0, 0.5, 1, 2, 5]

# Keep the full set of controls active in every test.
sensitivity_rows = []

# Change only the timing threshold.
for minutes in timing_thresholds:
    checks = get_control_checks(
        baseline_data, timing_limit=minutes, duplicate_limit=1
    )
    prediction = ~checks.all(axis=1)
    row = evaluate_predictions(
        "Timing limit: " + str(minutes) + " minutes",
        actual_abnormal, prediction
    )
    row["Control Tested"] = "Timing"
    row["Timing Limit (min)"] = minutes
    row["Duplicate Limit (min)"] = 1
    sensitivity_rows.append(row)

# Change only the duplicate threshold.
for minutes in duplicate_thresholds:
    checks = get_control_checks(
        baseline_data, timing_limit=15, duplicate_limit=minutes
    )
    prediction = ~checks.all(axis=1)
    row = evaluate_predictions(
        "Duplicate limit: " + str(minutes) + " minutes",
        actual_abnormal, prediction
    )
    row["Control Tested"] = "Duplicate"
    row["Timing Limit (min)"] = 15
    row["Duplicate Limit (min)"] = minutes
    sensitivity_rows.append(row)

# Review the sensitivity results.
sensitivity_results = pd.DataFrame(sensitivity_rows)
print("H1 threshold sensitivity:")
print(sensitivity_results[
    ["Control Tested", "Timing Limit (min)", "Duplicate Limit (min)",
     "True Positives", "False Positives", "False Negatives", "Recall"]
].round(4).to_string(index=False))

# Verify that the reference threshold reproduces the full result.
reference = sensitivity_results[
    (sensitivity_results["Control Tested"] == "Timing")
    & (sensitivity_results["Timing Limit (min)"] == 15)
].iloc[0]
assert int(reference["True Positives"]) == full_result["True Positives"]
print("\nStep 4 complete.")

H1 threshold sensitivity:
Control Tested  Timing Limit (min)  Duplicate Limit (min)  True Positives  False Positives  False Negatives  Recall
        Timing                   5                    1.0            1275                0               75  0.9444
        Timing                  10                    1.0            1275                0               75  0.9444
        Timing                  15                    1.0            1275                0               75  0.9444
        Timing                  20                    1.0            1050                0              300  0.7778
        Timing                  25                    1.0            1050                0              300  0.7778
        Timing                  30                    1.0            1050                0              300  0.7778
     Duplicate                  15                    0.0            1050                0              300  0.7778
     Duplicate                  15            

Step 5: Evaluate Legitimate Operational Variations

A reproducible sample of 300 originally normal scans is used for a controlled timing stress test. Their expected timestamps are shifted by 5, 10, 20, or 25 minutes, while actual scans and original labels remain unchanged. For this hypothetical test, those records are assumed to be legitimate despite the expectation mismatch. This assumption is not evidence of real airport operating conditions.

In [5]:
# Step 5: Evaluate controlled variations in normal scan timing

# Select one normal scan per bag to limit repeated sampling.
normal_records = baseline_data[
    baseline_data["final_label"].eq("Normal")
]
normal_candidates = normal_records.groupby(
    "bag_id", sort=False, group_keys=False
).sample(n=1, random_state=42)

# Select 300 unique baggage journeys with a reproducible random seed.
selected_normal = normal_candidates.sample(
    n=min(300, len(normal_candidates)), random_state=42
)
selected_indices = selected_normal.index

# Define hypothetical shifts in scheduled expected scan time.
expected_time_offsets = [5, 10, 20, 25]
variation_rows = []
variation_details = []

# Test each offset on a separate working copy of the same 300 normal scans.
for offset in expected_time_offsets:
    # Make a fresh working copy, never update the input dataset.
    altered_data = baseline_data.copy()

    # Make the original expected time earlier than the unchanged actual time.
    # Original labels are retained under a simulated-legitimate assumption.
    altered_data.loc[selected_indices, "expected_timestamp"] = (
        altered_data.loc[selected_indices, "expected_timestamp"]
        - pd.Timedelta(minutes=offset)
    )

    # Reapply the original 15-minute and one-minute thresholds.
    altered_checks = get_control_checks(
        altered_data, timing_limit=15, duplicate_limit=1
    )
    altered_prediction = ~altered_checks.all(axis=1)

    # Count alerts among records deliberately treated as normal.
    selected_prediction = altered_prediction.loc[selected_indices]
    selected_truth = actual_abnormal.loc[selected_indices]
    false_alert_count = int((selected_prediction & ~selected_truth).sum())

    # Calculate performance across all unchanged original labels.
    row = evaluate_predictions(
        "Assumed-normal expected-time offset: " + str(offset) + " min",
        actual_abnormal, altered_prediction
    )
    row["Offset (min)"] = offset
    row["Sampled Normal Scans"] = len(selected_indices)
    row["Alerts in Sample"] = false_alert_count
    row["Sample False Alert Rate"] = false_alert_count / len(selected_indices)
    variation_rows.append(row)

    # Save a compact audit record for each selected scan.
    for i in selected_indices:
        variation_details.append({
            "Offset (min)": offset,
            "Scan ID": baseline_data.at[i, "scan_id"],
            "Bag ID": baseline_data.at[i, "bag_id"],
            "Original Label": baseline_data.at[i, "final_label"],
            "Flagged Under Variation": bool(altered_prediction.at[i]),
        })

# Display the change in false alerts under each hypothetical assumption.
variation_results = pd.DataFrame(variation_rows)
variation_audit = pd.DataFrame(variation_details)
print("Simulated normal-scan timing variations:")
print(variation_results[
    ["Offset (min)", "Sampled Normal Scans", "Alerts in Sample",
     "Sample False Alert Rate", "False Positives", "Recall"]
].round(4).to_string(index=False))
print("\nStep 5 complete. Original data unchanged.")

Simulated normal-scan timing variations:
 Offset (min)  Sampled Normal Scans  Alerts in Sample  Sample False Alert Rate  False Positives  Recall
            5                   300                 0                      0.0                0  0.9444
           10                   300                 0                      0.0                0  0.9444
           20                   300               300                      1.0              300  0.9444
           25                   300               300                      1.0              300  0.9444

Step 5 complete. Original data unchanged.


Step 6: Compare Detection Performance and False Alerts

The original H1 decisions, individual control tests, threshold changes, and simulated legitimate timing variations are summarized. The analysis distinguishes scan-level false negatives from missing-scan journey tests completed separately in Notebook 03.

In [6]:
# Step 6: Compare detection performance and false alerts

# Combine the original comparison with supplemental scenario results.
all_scenario_results = pd.concat([
    baseline_comparison.assign(Test="Original H1 baseline"),
    control_contribution.assign(Test="Control removal"),
    sensitivity_results.assign(Test="Threshold sensitivity"),
    variation_results.assign(Test="Assumed-normal timing variation")
], ignore_index=True, sort=False)

# Calculate the improvement from traditional rules to Zero Trust.
trad_recall = float(baseline_comparison.loc[
    baseline_comparison["Scenario"] == "Traditional rules", "Recall"
].iloc[0])
zta_recall = float(baseline_comparison.loc[
    baseline_comparison["Scenario"] == "Zero Trust", "Recall"
].iloc[0])
print("Original recall difference (percentage points):",
      round((zta_recall - trad_recall) * 100, 2))

# Identify the original scan statuses missed by Zero Trust.
missed_scans = baseline_data.loc[
    actual_abnormal & ~zta_prediction,
    ["scan_status", "scan_id", "bag_id"]
].copy()
missed_status_counts = (
    missed_scans["scan_status"].value_counts()
    .rename_axis("Scan Status")
    .reset_index(name="Missed Scans")
)
print("\nOriginal abnormal scans missed by Zero Trust:")
print(missed_status_counts.to_string(index=False))

# Display the complete original comparison.
print("\nOriginal security method comparison:")
print(baseline_comparison[
    ["Scenario", "True Positives", "True Negatives", "False Positives",
     "False Negatives", "Precision", "Recall", "F1 Score"]
].round(4).to_string(index=False))

# Explain what the supplemental tests can and cannot establish.
print("\nInterpretation boundaries:")
print("- All H1 scenarios reuse the same simulated RFID source file.")
print("- Threshold changes test implemented rules, not new independent data.")
print("- Simulated normal variations assume legitimacy; they are not airport observations.")
print("- Missing scans were also assessed at the journey level in Notebook 03.")
print("Step 6 complete.")

Original recall difference (percentage points): 33.33

Original abnormal scans missed by Zero Trust:
      Scan Status  Missed Scans
Denial of Service            75

Original security method comparison:
         Scenario  True Positives  True Negatives  False Positives  False Negatives  Precision  Recall  F1 Score
Traditional rules             825           30225                0              525        1.0  0.6111    0.7586
       Zero Trust            1275           30225                0               75        1.0  0.9444    0.9714

Interpretation boundaries:
- All H1 scenarios reuse the same simulated RFID source file.
- Threshold changes test implemented rules, not new independent data.
- Simulated normal variations assume legitimacy; they are not airport observations.
- Missing scans were also assessed at the journey level in Notebook 03.
Step 6 complete.


Step 7: Save Results and Document Findings

The control comparison, threshold tests, simulated variations, and experiment settings are saved in a new timestamped folder under 07_H1_Results. No original H1 result files are replaced.

In [7]:
# Step 7: Save supplemental H1 results and experiment information

# Create a unique results folder for this execution.
run_time = datetime.now(timezone.utc)
run_id = run_time.strftime("%Y%m%d_%H%M%S_%f")
results_folder = (
    main_folder / "07_H1_Results"
    / ("Notebook05_H1_Control_Sensitivity_" + run_id)
)
results_folder.mkdir(parents=True, exist_ok=False)

# Save each research output without replacing earlier experiments.
baseline_comparison.to_csv(
    results_folder / "H1_NB05_Baseline_Comparison.csv", index=False
)
control_contribution.to_csv(
    results_folder / "H1_NB05_Control_Contribution.csv", index=False
)
sensitivity_results.to_csv(
    results_folder / "H1_NB05_Threshold_Sensitivity.csv", index=False
)
variation_results.to_csv(
    results_folder / "H1_NB05_Legitimate_Variation_Test.csv", index=False
)
variation_audit.to_csv(
    results_folder / "H1_NB05_Variation_Scan_Audit.csv", index=False
)
missed_status_counts.to_csv(
    results_folder / "H1_NB05_Missed_Scan_Conditions.csv", index=False
)
all_scenario_results.to_csv(
    results_folder / "H1_NB05_All_Scenario_Metrics.csv", index=False
)

# Describe the exact input and assumptions for reproducibility.
manifest = {
    "notebook": "05_Praxis_H1_Control_Contribution_and_Sensitivity_Analysis.ipynb",
    "hypothesis": "H1",
    "created_utc": run_time.isoformat(),
    "source_file": str(prepared_file.relative_to(main_folder)),
    "source_sha256": source_sha256,
    "source_records": int(len(baseline_data)),
    "source_baggage_journeys": int(baseline_data["bag_id"].nunique()),
    "security_controls": list(control_checks.columns),
    "reference_timing_minutes": 15,
    "reference_duplicate_minutes": 1,
    "tested_timing_thresholds": timing_thresholds,
    "tested_duplicate_thresholds": duplicate_thresholds,
    "variation_sample_seed": 42,
    "variation_normal_scans_sampled": int(len(selected_indices)),
    "variation_expected_time_offsets_minutes": expected_time_offsets,
    "method_note": (
        "The same original simulated RFID dataset was reused. "
        "Normal labels in variation tests are assumed to remain valid. "
        "Results are not independent field validation."
    ),
    "original_source_unchanged": True
}
with (results_folder / "H1_NB05_Experiment_Manifest.json").open("w") as output:
    json.dump(manifest, output, indent=2)

# Show the saved outputs.
print("Notebook 05 outputs saved:", results_folder)
for saved_file in sorted(results_folder.iterdir()):
    print(" -", saved_file.name)
print("Step 7 complete. Original H1 results preserved.")

Notebook 05 outputs saved: /workspaces/Aviation-Baggage-Cybersecurity-Research/07_H1_Results/Notebook05_H1_Control_Sensitivity_20261106_184354_516327
 - H1_NB05_All_Scenario_Metrics.csv
 - H1_NB05_Baseline_Comparison.csv
 - H1_NB05_Control_Contribution.csv
 - H1_NB05_Experiment_Manifest.json
 - H1_NB05_Legitimate_Variation_Test.csv
 - H1_NB05_Missed_Scan_Conditions.csv
 - H1_NB05_Threshold_Sensitivity.csv
 - H1_NB05_Variation_Scan_Audit.csv
Step 7 complete. Original H1 results preserved.
